# ETL — Star Schema Kepatuhan Pajak Kendaraan Bermotor Jawa Barat

Notebook ini menggabungkan seluruh paket `etl/` menjadi satu alur yang bisa dibaca berurutan:

**extract → transform → build star → validate → export**

Studi kasus: Badan Pendapatan Daerah (Bapenda) Provinsi Jawa Barat — kepatuhan daftar ulang pajak kendaraan bermotor dan estimasi potensi pendapatan daerah yang hilang. Sumbernya dataset agregat Open Data Jabar.

Prinsip yang dipegang (mengikuti brief proyek):

- **Extract manual.** Portal ada di belakang Cloudflare dan menolak bot, jadi CSV diunduh manual ke `data/raw/`. Notebook ini tidak pernah mengakses jaringan.
- **Idempoten.** Menjalankan ulang menghasilkan output identik — surrogate key stabil dan urutan deterministik.
- **Path relatif.** Semua path dihitung dari root repo.

> Jalankan notebook ini **dari root repo** supaya `ROOT_PROYEK = Path.cwd()` menunjuk ke folder yang benar (berisi `config/` dan `data/`).

## Setup — import & konstanta

Satu-satunya sel impor untuk seluruh pipeline, plus konstanta path dan URL portal. Tidak ada library jaringan di sini — sesuai batasan extract manual.

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import yaml

__version__ = "0.1.0"

ROOT_PROYEK: Path = Path.cwd()
KONFIG_DEFAULT: Path = ROOT_PROYEK / "config" / "sources.yaml"
DIR_RAW_DEFAULT: Path = ROOT_PROYEK / "data" / "raw"
DIR_PROCESSED_DEFAULT: Path = ROOT_PROYEK / "data" / "processed"

URL_KATALOG = "https://opendata.jabarprov.go.id/id/dataset"
URL_BAPENDA = "https://bapenda.jabarprov.go.id/open-data-jabar/"

## 1 · Extract — baca CSV mentah dari `data/raw/`

Membaca CSV mentah apa adanya (semua kolom sebagai string; pembersihan tipe ditunda ke tahap transform supaya pemisah ribuan/desimal tidak dirusak). Daftar sumber dibaca dari `config/sources.yaml`.

Kalau ada file **wajib** yang belum ada, pipeline berhenti cepat dengan pesan yang menyebut file mana yang kurang beserta nama dataset aslinya di portal — bukan mencoba mengunduh sendiri.

In [ ]:
class BerkasMentahHilangError(FileNotFoundError):
    pass


class KonfigurasiTidakValidError(ValueError):
    pass


@dataclass(frozen=True)
class TabelMentah:
    alias: str
    path: Path
    df: pd.DataFrame
    konfig: dict[str, Any]

    @property
    def jumlah_baris(self) -> int:
        return len(self.df)


def muat_konfigurasi(path_konfig: str | Path | None = None) -> dict[str, Any]:
    path = Path(path_konfig) if path_konfig else KONFIG_DEFAULT
    if not path.exists():
        raise KonfigurasiTidakValidError(
            f"File konfigurasi tidak ditemukan: {_relatif(path)}"
        )

    with path.open("r", encoding="utf-8") as f:
        konfig = yaml.safe_load(f) or {}

    if not isinstance(konfig, dict):
        raise KonfigurasiTidakValidError(
            f"{_relatif(path)} harus berupa mapping YAML di level teratas."
        )
    if "sumber" not in konfig or not isinstance(konfig["sumber"], dict):
        raise KonfigurasiTidakValidError(
            f"{_relatif(path)} wajib punya blok `sumber:` berisi minimal satu dataset."
        )

    konfig.setdefault("meta", {})
    konfig.setdefault("alias_kolom", {})
    konfig.setdefault("normalisasi_teks", {})
    konfig.setdefault("monetisasi", {})
    konfig.setdefault("sumber_opsional", {})
    return konfig


def path_sumber(konfig_sumber: dict[str, Any], root: Path | None = None) -> Path:
    root = root or ROOT_PROYEK
    berkas = Path(konfig_sumber.get("file", ""))
    return berkas if berkas.is_absolute() else root / berkas


def periksa_berkas_mentah(
    konfig: dict[str, Any], root: Path | None = None
) -> dict[str, Path]:
    root = root or ROOT_PROYEK
    tersedia: dict[str, Path] = {}
    hilang: list[tuple[str, Path, str]] = []

    for alias, ksum in konfig["sumber"].items():
        path = path_sumber(ksum, root)
        if not ksum.get("wajib", True):
            if path.exists():
                tersedia[alias] = path
            continue
        if path.exists():
            tersedia[alias] = path
        else:
            hilang.append((alias, path, str(ksum.get("deskripsi", "")).strip()))

    if hilang:
        raise BerkasMentahHilangError(_pesan_berkas_hilang(hilang, root))
    return tersedia


def _pesan_berkas_hilang(
    hilang: list[tuple[str, Path, str]], root: Path
) -> str:
    baris = [
        "",
        "=" * 78,
        "ETL DIHENTIKAN: file CSV mentah belum lengkap di data/raw/.",
        "=" * 78,
        "",
        f"Ada {len(hilang)} file wajib yang belum ada:",
        "",
    ]
    for alias, path, deskripsi in hilang:
        baris.append(f"  [-] {_relatif(path, root)}")
        baris.append(f"      alias    : {alias}")
        if deskripsi:
            baris.append(f"      dataset  : {deskripsi}")
        baris.append("")
    baris += [
        "Cara memperbaiki (extract MANUAL -- portal mem-block bot, lihat README):",
        "  1. Buka katalog dataset lewat browser:",
        f"       {URL_KATALOG}",
        f"       {URL_BAPENDA}",
        "  2. Cari dataset sesuai deskripsi di atas, klik Unduh -> CSV",
        "     (portal memberi .zip berisi CSV + metadata PDF).",
        "  3. Ekstrak .zip, ganti nama CSV-nya persis seperti path di atas,",
        "     lalu taruh di data/raw/.",
        "  4. Jalankan ulang pipeline dari notebook ini.",
        "",
        "Catatan: ETL ini TIDAK mengunduh sendiri dari portal (Cloudflare 403).",
        "=" * 78,
    ]
    return "\n".join(baris)


def baca_csv(
    path: Path, encoding: str = "utf-8", sep: str | None = None
) -> pd.DataFrame:
    opsi: dict[str, Any] = {"dtype": str, "keep_default_na": True}
    if sep in (None, "auto", ""):
        opsi["sep"] = None
        opsi["engine"] = "python"
    else:
        opsi["sep"] = sep

    for enc in _kandidat_encoding(encoding):
        try:
            df = pd.read_csv(path, encoding=enc, **opsi)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise UnicodeDecodeError(
            "utf-8", b"", 0, 1, f"Gagal membaca {_relatif(path)} dengan encoding manapun"
        )

    df.attrs["sumber_file"] = str(path)
    return df


def _kandidat_encoding(encoding: str) -> list[str]:
    kandidat = [encoding or "utf-8", "utf-8-sig", "cp1252", "latin-1"]
    hasil: list[str] = []
    for enc in kandidat:
        if enc not in hasil:
            hasil.append(enc)
    return hasil


def ekstrak(
    konfig: dict[str, Any], root: Path | None = None
) -> dict[str, TabelMentah]:
    root = root or ROOT_PROYEK
    tersedia = periksa_berkas_mentah(konfig, root)

    meta = konfig.get("meta", {}) or {}
    encoding = str(meta.get("encoding", "utf-8"))
    sep = meta.get("sep", "auto")

    hasil: dict[str, TabelMentah] = {}
    for alias, path in tersedia.items():
        ksum = konfig["sumber"][alias]
        df = baca_csv(path, encoding=encoding, sep=sep)
        hasil[alias] = TabelMentah(alias=alias, path=path, df=df, konfig=ksum)
    return hasil


def ringkas_ekstrak(tabel: dict[str, TabelMentah], root: Path | None = None) -> str:
    root = root or ROOT_PROYEK
    baris = []
    for alias, t in tabel.items():
        baris.append(
            f"  {alias:<22} {t.jumlah_baris:>8,} baris x {len(t.df.columns):>2} kolom"
            f"  <- {_relatif(t.path, root)}"
        )
    return "\n".join(baris)


def _relatif(path: Path, root: Path | None = None) -> str:
    root = root or ROOT_PROYEK
    try:
        return Path(path).resolve().relative_to(Path(root).resolve()).as_posix()
    except ValueError:
        return Path(path).as_posix()

## 2 · Transform — bersihkan, unpivot, samakan grain

Menerapkan aturan transform pada brief:

- **Unpivot** data wide (`pandas.melt`) menjadi long bila kategori tersimpan sebagai kolom.
- **Standarisasi teks** dimensi: trim, rapikan spasi, Title Case, dan pemetaan ejaan yang tak konsisten.
- **Tipe data:** `tahun` → integer; measure → numerik (pemisah ribuan dibuang, koma desimal ditangani).
- **Samakan grain** fact `(tahun, cabang_pelayanan, jenis_kendaraan, fungsi_kendaraan)`; baris agregat/total dibuang agar measure tidak terhitung ganda.

Semua fungsi di sini murni: tidak menulis file, tidak menyentuh jaringan.

In [ ]:
NILAI_AGREGAT: frozenset[str] = frozenset(
    {
        "jumlah",
        "total",
        "jumlah total",
        "total jumlah",
        "grand total",
        "semua",
        "seluruhnya",
        "jumlah keseluruhan",
    }
)

NILAI_TIDAK_DIRINCI = "Tidak Dirinci"

KOLOM_ATRIBUT_WILAYAH = (
    "kode_provinsi",
    "nama_provinsi",
    "kode_kabupaten_kota",
    "nama_kabupaten_kota",
)


class GrainTidakLengkapError(ValueError):
    pass


@dataclass
class HasilTransform:
    alias: str
    df: pd.DataFrame
    measure: str
    keys: list[str]
    catatan: list[str] = field(default_factory=list)
    nilai_belum_dipetakan: dict[str, list[str]] = field(default_factory=dict)

    def catat(self, pesan: str) -> None:
        self.catatan.append(f"[{self.alias}] {pesan}")


def normalisasi_nama_kolom(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_slug_kolom(str(k)) for k in df.columns]
    return df


def _slug_kolom(nama: str) -> str:
    nama = nama.replace("﻿", "").strip().lower()
    nama = re.sub(r"[^\w]+", "_", nama, flags=re.UNICODE)
    return re.sub(r"_+", "_", nama).strip("_")


def bangun_peta_alias(alias_kolom: Mapping[str, Sequence[str]]) -> dict[str, str]:
    peta: dict[str, str] = {}
    for baku, varian in (alias_kolom or {}).items():
        peta.setdefault(_slug_kolom(baku), baku)
        for v in varian or []:
            peta.setdefault(_slug_kolom(str(v)), baku)
    return peta


def petakan_kolom(
    df: pd.DataFrame,
    peta_alias: Mapping[str, str],
    rename_eksplisit: Mapping[str, str] | None = None,
) -> pd.DataFrame:
    rename_eksplisit = {
        _slug_kolom(k): v for k, v in (rename_eksplisit or {}).items()
    }
    mapping: dict[str, str] = {}
    terpakai: set[str] = set()
    for kolom in df.columns:
        baku = rename_eksplisit.get(kolom) or peta_alias.get(kolom)
        if baku and baku not in terpakai:
            mapping[kolom] = baku
            terpakai.add(baku)
    return df.rename(columns=mapping)


def unpivot_wide(
    df: pd.DataFrame,
    id_vars: Sequence[str],
    var_name: str,
    value_name: str,
) -> pd.DataFrame:
    id_ada = [k for k in id_vars if k in df.columns]
    value_vars = [k for k in df.columns if k not in id_ada]
    if not value_vars:
        raise GrainTidakLengkapError(
            f"Tidak ada kolom kategori untuk di-unpivot; kolom tersedia: {list(df.columns)}"
        )
    panjang = df.melt(
        id_vars=id_ada,
        value_vars=value_vars,
        var_name=var_name,
        value_name=value_name,
    )
    return panjang.reset_index(drop=True)


def tentukan_bentuk(
    df: pd.DataFrame, bentuk: str, measure_sumber: Sequence[str]
) -> str:
    bentuk = (bentuk or "auto").lower()
    if bentuk in {"long", "wide"}:
        return bentuk
    return "long" if _cari_kolom_measure(df, measure_sumber) else "wide"


def _cari_kolom_measure(
    df: pd.DataFrame, measure_sumber: Sequence[str]
) -> str | None:
    for kandidat in measure_sumber or []:
        slug = _slug_kolom(str(kandidat))
        if slug in df.columns:
            return slug
    return None


def bersihkan_teks(seri: pd.Series) -> pd.Series:
    hasil = seri.astype("string")
    hasil = hasil.str.replace(r"[_\s]+", " ", regex=True).str.strip()
    hasil = hasil.replace({"": pd.NA, "-": pd.NA, "nan": pd.NA, "NaN": pd.NA})
    return hasil.str.title()


def bangun_peta_nilai(peta_yaml: Mapping[str, Sequence[str]]) -> dict[str, str]:
    peta: dict[str, str] = {}
    for baku, varian in (peta_yaml or {}).items():
        peta.setdefault(_kunci_nilai(baku), baku)
        for v in varian or []:
            peta.setdefault(_kunci_nilai(str(v)), baku)
    return peta


def _kunci_nilai(nilai: str) -> str:
    return re.sub(r"[\s_]+", " ", str(nilai)).strip().lower()


def terapkan_peta_nilai(
    seri: pd.Series, peta_nilai: Mapping[str, str]
) -> tuple[pd.Series, list[str]]:
    bersih = bersihkan_teks(seri)
    if not peta_nilai:
        return bersih, []

    kunci = bersih.map(lambda v: _kunci_nilai(v) if pd.notna(v) else v)
    hasil = kunci.map(lambda k: peta_nilai.get(k) if pd.notna(k) else k)
    belum = sorted(
        {
            str(asli)
            for asli, baku in zip(bersih, hasil)
            if pd.notna(asli) and pd.isna(baku)
        }
    )
    hasil = hasil.astype("string").fillna(bersih)
    return hasil, belum


_POLA_TAHUN = r"((?:19|20)\d{2})"


def ke_tahun(seri: pd.Series) -> pd.Series:
    teks = seri.astype("string").str.strip()
    cocok = teks.str.extract(_POLA_TAHUN, expand=False)
    return pd.to_numeric(cocok, errors="coerce").astype("Int64")


def ke_numerik(seri: pd.Series) -> pd.Series:
    return seri.map(_parse_angka).astype("float64")


def _parse_angka(nilai: Any) -> float:
    if nilai is None or (isinstance(nilai, float) and np.isnan(nilai)):
        return np.nan
    if isinstance(nilai, (int, np.integer, float, np.floating)):
        return float(nilai)

    teks = str(nilai).strip()
    if teks == "" or teks.lower() in {"na", "nan", "none", "null", "-", "--"}:
        return np.nan

    negatif = teks.startswith("(") and teks.endswith(")")
    teks = re.sub(r"(?i)\b(rp|idr|unit|kendaraan)\b", "", teks)
    teks = re.sub(r"[^\d,.\-]", "", teks)
    if teks in {"", "-", ".", ","}:
        return np.nan
    if teks.startswith("-"):
        negatif = True
    teks = teks.lstrip("+-")

    ada_titik, ada_koma = "." in teks, "," in teks
    if ada_titik and ada_koma:
        if teks.rfind(",") > teks.rfind("."):
            teks = teks.replace(".", "").replace(",", ".")
        else:
            teks = teks.replace(",", "")
    elif ada_koma:
        teks = teks.replace(",", "") if _pola_ribuan(teks, ",") else teks.replace(",", ".")
    elif ada_titik:
        if _pola_ribuan(teks, "."):
            teks = teks.replace(".", "")

    try:
        angka = float(teks)
    except ValueError:
        return np.nan
    return -angka if negatif else angka


def _pola_ribuan(teks: str, pemisah: str) -> bool:
    bagian = teks.split(pemisah)
    if len(bagian) < 2:
        return False
    if len(bagian) > 2:
        return all(len(b) == 3 for b in bagian[1:])
    return len(bagian[1]) == 3 and len(bagian[0]) <= 3


def transform_sumber(
    df_mentah: pd.DataFrame,
    alias: str,
    konfig_sumber: Mapping[str, Any],
    konfig: Mapping[str, Any],
) -> HasilTransform:
    keys = list(konfig_sumber.get("keys") or [])
    measure = str(konfig_sumber["measure"])
    measure_sumber = list(konfig_sumber.get("measure_sumber") or [])
    peta_alias = bangun_peta_alias(konfig.get("alias_kolom", {}))
    peta_teks_semua = konfig.get("normalisasi_teks", {}) or {}

    hasil = HasilTransform(alias=alias, df=pd.DataFrame(), measure=measure, keys=keys)

    df = normalisasi_nama_kolom(df_mentah)
    df = petakan_kolom(df, peta_alias, konfig_sumber.get("rename"))

    bentuk = tentukan_bentuk(df, str(konfig_sumber.get("bentuk", "auto")), measure_sumber)
    atribut = [k for k in KOLOM_ATRIBUT_WILAYAH if k in df.columns]

    if bentuk == "long":
        kolom_measure = _cari_kolom_measure(df, measure_sumber)
        if kolom_measure is None:
            raise GrainTidakLengkapError(
                f"[{alias}] bentuk 'long' tapi kolom measure {measure_sumber} tidak ada. "
                f"Kolom tersedia: {list(df.columns)}. Perbaiki `measure_sumber` di config/sources.yaml."
            )
        df = df.rename(columns={kolom_measure: measure})
        hasil.catat(f"bentuk=long, kolom measure sumber='{kolom_measure}'")
    else:
        unpivot = konfig_sumber.get("unpivot") or {}
        id_vars = list(unpivot.get("id_vars") or [k for k in keys if k != unpivot.get("var_name")])
        id_vars = list(dict.fromkeys(id_vars + atribut + ["satuan"]))
        var_name = str(unpivot.get("var_name") or "kategori")
        df = unpivot_wide(df, id_vars=id_vars, var_name=var_name, value_name=measure)
        hasil.catat(f"bentuk=wide, di-unpivot menjadi kolom '{var_name}'")

    for k in keys:
        if k in df.columns:
            continue
        if k == "tahun":
            raise GrainTidakLengkapError(
                f"[{alias}] kolom 'tahun' tidak ditemukan. Kolom tersedia: {list(df.columns)}. "
                "Tambahkan alias yang benar di config/sources.yaml -> alias_kolom.tahun."
            )
        df[k] = NILAI_TIDAK_DIRINCI
        hasil.catat(
            f"kolom grain '{k}' tidak ada di sumber -> diisi '{NILAI_TIDAK_DIRINCI}' (ASUMSI)"
        )

    df["tahun"] = ke_tahun(df["tahun"])
    df[measure] = ke_numerik(df[measure])

    for kolom in [k for k in keys if k != "tahun"] + atribut:
        if kolom in {"kode_provinsi", "kode_kabupaten_kota"}:
            df[kolom] = df[kolom].astype("string").str.strip()
            continue
        seri, belum = terapkan_peta_nilai(df[kolom], bangun_peta_nilai(peta_teks_semua.get(kolom, {})))
        df[kolom] = seri
        if belum:
            hasil.nilai_belum_dipetakan[kolom] = belum

    sebelum = len(df)
    df = df[df["tahun"].notna()]
    if (dibuang := sebelum - len(df)) > 0:
        hasil.catat(f"{dibuang:,} baris dibuang: 'tahun' tidak bisa diparse")

    for kolom in [k for k in keys if k != "tahun"]:
        mask = df[kolom].map(lambda v: _kunci_nilai(v) in NILAI_AGREGAT if pd.notna(v) else False)
        if (n := int(mask.sum())) > 0:
            df = df[~mask]
            hasil.catat(f"{n:,} baris agregat/total dibuang dari kolom '{kolom}'")

    kolom_akhir = keys + [measure] + atribut
    df = df[[k for k in kolom_akhir if k in df.columns]]
    df, n_gabung = agregasi_ke_grain(df, keys, measure, atribut)
    if n_gabung:
        hasil.catat(f"{n_gabung:,} baris duplikat pada grain digabung (sum measure)")

    df = df.sort_values(keys, kind="mergesort").reset_index(drop=True)
    hasil.df = df
    hasil.catat(f"hasil akhir: {len(df):,} baris, grain={tuple(keys)}")
    return hasil


def agregasi_ke_grain(
    df: pd.DataFrame,
    keys: Sequence[str],
    measure: str,
    atribut: Sequence[str] = (),
) -> tuple[pd.DataFrame, int]:
    keys = list(keys)
    atribut = [a for a in atribut if a in df.columns and a not in keys]
    if df.empty:
        return df.reset_index(drop=True), 0

    agg: dict[str, Any] = {measure: "sum"}
    for a in atribut:
        agg[a] = "first"
    hasil = df.groupby(keys, dropna=False, as_index=False, sort=True).agg(agg)
    return hasil, len(df) - len(hasil)


def transform_semua(
    tabel_mentah: Mapping[str, Any], konfig: Mapping[str, Any]
) -> dict[str, HasilTransform]:
    hasil: dict[str, HasilTransform] = {}
    for alias, tabel in tabel_mentah.items():
        df = tabel.df if hasattr(tabel, "df") else tabel
        ksum = tabel.konfig if hasattr(tabel, "konfig") else konfig["sumber"][alias]
        hasil[alias] = transform_sumber(df, alias, ksum, konfig)
    return hasil


def kumpulkan_catatan(hasil: Mapping[str, HasilTransform]) -> list[str]:
    catatan: list[str] = []
    for h in hasil.values():
        catatan.extend(h.catatan)
        for kolom, nilai in h.nilai_belum_dipetakan.items():
            contoh = ", ".join(nilai[:8])
            lanjut = " ..." if len(nilai) > 8 else ""
            catatan.append(
                f"[{h.alias}] {len(nilai)} nilai '{kolom}' belum dipetakan di "
                f"normalisasi_teks: {contoh}{lanjut}"
            )
    return catatan


def daftar_unik(seri: Iterable[Any]) -> list[str]:
    nilai = {str(v) for v in seri if pd.notna(v)}
    return sorted(nilai)

## 3 · Build Star — dimensi, surrogate key, fact, measure turunan

Membangun star schema target:

- **Dimensi** `dim_waktu`, `dim_wilayah`, `dim_jenis_kendaraan`, `dim_fungsi_kendaraan` dengan **surrogate key** integer yang stabil (dibangun dari natural key terurut → idempoten).
- **Fact** `fakta_kepatuhan_kendaraan` dari outer-join tiga dataset count inti pada grain.
- **Measure turunan:** `jumlah_daftar_ulang` (= total − tidak − belum, di-floor ke 0), `tingkat_kepatuhan` (rasio 0–1, null bila penyebut tak diketahui), dan `estimasi_pajak_hilang` (dari rata-rata pajak per kendaraan per tahun).

In [ ]:
GRAIN_FACT: tuple[str, ...] = (
    "tahun",
    "cabang_pelayanan",
    "jenis_kendaraan",
    "fungsi_kendaraan",
)

MEASURE_INTI: dict[str, str] = {
    "kendaraan_total": "jumlah_kendaraan_total",
    "tidak_daftar_ulang": "jumlah_tidak_daftar_ulang",
    "belum_daftar_ulang": "jumlah_belum_daftar_ulang",
}

KOLOM_FACT: tuple[str, ...] = (
    "waktu_id",
    "wilayah_id",
    "jenis_kendaraan_id",
    "fungsi_kendaraan_id",
    "tahun",
    "cabang_pelayanan",
    "jenis_kendaraan",
    "fungsi_kendaraan",
    "jumlah_kendaraan_total",
    "jumlah_tidak_daftar_ulang",
    "jumlah_belum_daftar_ulang",
    "jumlah_daftar_ulang",
    "tingkat_kepatuhan",
    "estimasi_pajak_hilang",
    "rata_pajak_per_kendaraan",
)


@dataclass
class StarSchema:
    tabel: dict[str, pd.DataFrame] = field(default_factory=dict)
    catatan: list[str] = field(default_factory=list)
    metrik: dict[str, Any] = field(default_factory=dict)

    def catat(self, pesan: str) -> None:
        self.catatan.append(f"[build] {pesan}")

    @property
    def fact(self) -> pd.DataFrame:
        return self.tabel["fakta_kepatuhan_kendaraan"]


def buat_dimensi(
    nilai: Sequence[Any],
    kolom_natural: str,
    kolom_id: str,
    atribut: pd.DataFrame | None = None,
) -> pd.DataFrame:
    unik = pd.Series(list(nilai)).dropna().drop_duplicates()
    unik = unik.sort_values(kind="mergesort").reset_index(drop=True)

    dim = pd.DataFrame({kolom_natural: unik})
    dim.insert(0, kolom_id, np.arange(1, len(dim) + 1, dtype="int64"))

    if atribut is not None and not atribut.empty:
        dim = dim.merge(atribut, on=kolom_natural, how="left")
    return dim


def bangun_dim_waktu(tahun: Sequence[Any]) -> pd.DataFrame:
    seri = pd.to_numeric(pd.Series(list(tahun)), errors="coerce").dropna().astype("int64")
    dim = buat_dimensi(seri, "tahun", "waktu_id")
    dim["tahun"] = dim["tahun"].astype("int64")
    dim["dekade"] = (dim["tahun"] // 10 * 10).astype("int64")
    dim["label_tahun"] = dim["tahun"].astype(str)
    return dim


def bangun_dim_wilayah(
    cabang: Sequence[Any], atribut: pd.DataFrame | None = None
) -> pd.DataFrame:
    dim = buat_dimensi(cabang, "cabang_pelayanan", "wilayah_id", atribut)
    for kolom in ("kode_kabupaten_kota", "nama_kabupaten_kota", "nama_provinsi"):
        if kolom not in dim.columns:
            dim[kolom] = pd.NA
    dim["nama_provinsi"] = dim["nama_provinsi"].fillna("Jawa Barat")
    return dim[
        [
            "wilayah_id",
            "cabang_pelayanan",
            "kode_kabupaten_kota",
            "nama_kabupaten_kota",
            "nama_provinsi",
        ]
    ]


def kumpulkan_atribut_wilayah(
    hasil: Mapping[str, HasilTransform]
) -> pd.DataFrame:
    potongan: list[pd.DataFrame] = []
    kolom = ["cabang_pelayanan", "kode_kabupaten_kota", "nama_kabupaten_kota", "nama_provinsi"]
    for h in hasil.values():
        ada = [k for k in kolom if k in h.df.columns]
        if "cabang_pelayanan" in ada and len(ada) > 1:
            potongan.append(h.df[ada].dropna(subset=["cabang_pelayanan"]))
    if not potongan:
        return pd.DataFrame(columns=kolom)

    gabung = pd.concat(potongan, ignore_index=True).drop_duplicates()
    gabung = gabung.sort_values(list(gabung.columns), kind="mergesort")
    return gabung.groupby("cabang_pelayanan", as_index=False, sort=True).first()


def gabung_measure_inti(
    hasil: Mapping[str, HasilTransform], star: StarSchema | None = None
) -> pd.DataFrame:
    grain = list(GRAIN_FACT)
    gabungan: pd.DataFrame | None = None

    for alias, kolom_measure in MEASURE_INTI.items():
        h = hasil.get(alias)
        if h is None:
            raise KeyError(
                f"Sumber inti '{alias}' tidak ada pada hasil transform; "
                "cek config/sources.yaml -> sumber."
            )
        potong = h.df[[*grain, kolom_measure]].copy()
        gabungan = potong if gabungan is None else gabungan.merge(potong, on=grain, how="outer")

    assert gabungan is not None
    gabungan = gabungan.sort_values(grain, kind="mergesort").reset_index(drop=True)

    for kolom in ("jumlah_tidak_daftar_ulang", "jumlah_belum_daftar_ulang"):
        n_null = int(gabungan[kolom].isna().sum())
        if n_null and star is not None:
            star.catat(f"{n_null:,} sel '{kolom}' kosong -> diisi 0 (tidak ada kejadian)")
        gabungan[kolom] = gabungan[kolom].fillna(0.0)

    n_total_null = int(gabungan["jumlah_kendaraan_total"].isna().sum())
    if n_total_null and star is not None:
        star.catat(
            f"{n_total_null:,} baris tanpa 'jumlah_kendaraan_total' -> dibiarkan NaN "
            "(penyebut tidak diketahui, tingkat_kepatuhan jadi null)"
        )
    return gabungan


def hitung_rata_pajak_per_kendaraan(
    df_pajak: pd.DataFrame | None,
    df_fact: pd.DataFrame,
    monetisasi: Mapping[str, Any] | None = None,
    star: StarSchema | None = None,
) -> pd.DataFrame:
    monetisasi = monetisasi or {}
    pola_pajak = [str(p).lower() for p in monetisasi.get("komponen_pajak", ["pkb"])]
    pola_denda = [str(p).lower() for p in monetisasi.get("komponen_denda", ["denda"])]

    penyebut = (
        df_fact.groupby("tahun", as_index=False, sort=True)["jumlah_kendaraan_total"]
        .sum(min_count=1)
        .rename(columns={"jumlah_kendaraan_total": "kendaraan_terdaftar"})
    )

    if df_pajak is None or df_pajak.empty or "nilai_rupiah" not in df_pajak.columns:
        if star is not None:
            star.catat(
                "data pajak_denda tidak tersedia -> estimasi_pajak_hilang = NaN "
                "(rata_pajak_per_kendaraan tidak bisa dihitung)"
            )
        hasil = penyebut.copy()
        hasil["nilai_pajak_rupiah"] = np.nan
        hasil["rata_pajak_per_kendaraan"] = np.nan
        return hasil[
            ["tahun", "nilai_pajak_rupiah", "kendaraan_terdaftar", "rata_pajak_per_kendaraan"]
        ]

    jenis = df_pajak["jenis_pajak"].astype("string").str.lower().fillna("")
    is_denda = jenis.apply(lambda v: any(p in v for p in pola_denda))
    is_pajak = jenis.apply(lambda v: any(p in v for p in pola_pajak)) & ~is_denda

    if not bool(is_pajak.any()):
        if star is not None:
            star.catat(
                f"tidak ada baris pajak_denda yang cocok pola {pola_pajak} -> "
                f"memakai SELURUH baris non-denda sebagai komponen pajak (ASUMSI)"
            )
        is_pajak = ~is_denda

    pembilang = (
        df_pajak.loc[is_pajak]
        .groupby("tahun", as_index=False, sort=True)["nilai_rupiah"]
        .sum(min_count=1)
        .rename(columns={"nilai_rupiah": "nilai_pajak_rupiah"})
    )

    hasil = penyebut.merge(pembilang, on="tahun", how="left")
    hasil["rata_pajak_per_kendaraan"] = np.where(
        (hasil["kendaraan_terdaftar"] > 0) & hasil["nilai_pajak_rupiah"].notna(),
        hasil["nilai_pajak_rupiah"] / hasil["kendaraan_terdaftar"],
        np.nan,
    )

    kosong = hasil["rata_pajak_per_kendaraan"].isna()
    if bool(kosong.any()) and monetisasi.get("fallback_rata_global", True):
        rata_global = float(hasil["rata_pajak_per_kendaraan"].mean(skipna=True) or np.nan)
        if not np.isnan(rata_global):
            tahun_fallback = hasil.loc[kosong, "tahun"].tolist()
            hasil.loc[kosong, "rata_pajak_per_kendaraan"] = rata_global
            if star is not None:
                star.catat(
                    f"tahun {tahun_fallback} tidak punya data pajak -> memakai rata-rata "
                    f"global Rp {rata_global:,.0f}/kendaraan (ASUMSI, lihat data_dictionary)"
                )

    return hasil[
        ["tahun", "nilai_pajak_rupiah", "kendaraan_terdaftar", "rata_pajak_per_kendaraan"]
    ]


def hitung_measure_turunan(
    df: pd.DataFrame, star: StarSchema | None = None
) -> pd.DataFrame:
    df = df.copy()
    mentah = (
        df["jumlah_kendaraan_total"]
        - df["jumlah_tidak_daftar_ulang"]
        - df["jumlah_belum_daftar_ulang"]
    )
    n_negatif = int((mentah < 0).sum())
    df["jumlah_daftar_ulang"] = mentah.clip(lower=0)
    if n_negatif and star is not None:
        star.catat(
            f"{n_negatif:,} baris punya (total - tidak - belum) < 0 -> di-floor ke 0 "
            "(inkonsistensi sumber, aturan Bagian 7 butir 5)"
        )
    if star is not None:
        star.metrik["baris_difloor"] = n_negatif

    total = df["jumlah_kendaraan_total"]
    df["tingkat_kepatuhan"] = np.where(
        total.notna() & (total > 0), df["jumlah_daftar_ulang"] / total, np.nan
    )

    if "rata_pajak_per_kendaraan" in df.columns:
        df["estimasi_pajak_hilang"] = (
            df["jumlah_tidak_daftar_ulang"] * df["rata_pajak_per_kendaraan"]
        )
    else:
        df["estimasi_pajak_hilang"] = np.nan
    return df


def bangun_star(
    hasil: Mapping[str, HasilTransform], konfig: Mapping[str, Any] | None = None
) -> StarSchema:
    konfig = konfig or {}
    star = StarSchema()

    fact = gabung_measure_inti(hasil, star)

    dim_waktu = bangun_dim_waktu(fact["tahun"])
    dim_wilayah = bangun_dim_wilayah(
        fact["cabang_pelayanan"], kumpulkan_atribut_wilayah(hasil)
    )
    dim_jenis = buat_dimensi(
        fact["jenis_kendaraan"], "jenis_kendaraan", "jenis_kendaraan_id"
    )
    dim_fungsi = buat_dimensi(
        fact["fungsi_kendaraan"], "fungsi_kendaraan", "fungsi_kendaraan_id"
    )

    h_pajak = hasil.get("pajak_denda")
    tabel_rata = hitung_rata_pajak_per_kendaraan(
        h_pajak.df if h_pajak is not None else None,
        fact,
        konfig.get("monetisasi", {}),
        star,
    )
    fact["tahun"] = pd.to_numeric(fact["tahun"], errors="coerce").astype("Int64")
    fact = fact.merge(
        tabel_rata[["tahun", "rata_pajak_per_kendaraan"]].assign(
            tahun=lambda d: pd.to_numeric(d["tahun"], errors="coerce").astype("Int64")
        ),
        on="tahun",
        how="left",
    )

    fact = hitung_measure_turunan(fact, star)
    fact = fact.merge(dim_waktu[["waktu_id", "tahun"]].astype({"tahun": "Int64"}), on="tahun", how="left")
    fact = fact.merge(dim_wilayah[["wilayah_id", "cabang_pelayanan"]], on="cabang_pelayanan", how="left")
    fact = fact.merge(dim_jenis, on="jenis_kendaraan", how="left")
    fact = fact.merge(dim_fungsi, on="fungsi_kendaraan", how="left")

    fact = fact[[k for k in KOLOM_FACT if k in fact.columns]]
    fact = fact.sort_values(list(GRAIN_FACT), kind="mergesort").reset_index(drop=True)
    for kolom in ("waktu_id", "wilayah_id", "jenis_kendaraan_id", "fungsi_kendaraan_id"):
        fact[kolom] = fact[kolom].astype("Int64")

    star.tabel = {
        "dim_waktu": dim_waktu,
        "dim_wilayah": dim_wilayah,
        "dim_jenis_kendaraan": dim_jenis,
        "dim_fungsi_kendaraan": dim_fungsi,
        "fakta_kepatuhan_kendaraan": fact,
        "bantu_rata_pajak_tahunan": tabel_rata,
    }
    star.metrik["jumlah_baris"] = {k: len(v) for k, v in star.tabel.items()}
    star.catat(
        f"fact terbangun: {len(fact):,} baris, {fact['tahun'].nunique()} tahun, "
        f"{fact['cabang_pelayanan'].nunique()} cabang pelayanan"
    )
    if (fact["fungsi_kendaraan"] == NILAI_TIDAK_DIRINCI).all():
        star.catat(
            "seluruh 'fungsi_kendaraan' bernilai 'Tidak Dirinci' -- sumber tidak "
            "memuat dimensi ini (ASUMSI didokumentasikan di data_dictionary)"
        )
    return star

## 4 · Validate — cek kualitas star schema

Pemeriksaan kualitas sesuai brief.

- **Fatal** (menggagalkan ETL): FK fact tanpa pasangan di dim atau bernilai null, `tingkat_kepatuhan` di luar [0, 1], baris fact duplikat pada grain, dan dimensi kosong.
- **Warning** (dicatat, pipeline tetap lanjut): rentang `tahun` menyimpang dari ekspektasi, jumlah null per measure, dan jumlah baris yang di-floor.

In [ ]:
RELASI_FK: dict[str, tuple[str, str]] = {
    "waktu_id": ("dim_waktu", "waktu_id"),
    "wilayah_id": ("dim_wilayah", "wilayah_id"),
    "jenis_kendaraan_id": ("dim_jenis_kendaraan", "jenis_kendaraan_id"),
    "fungsi_kendaraan_id": ("dim_fungsi_kendaraan", "fungsi_kendaraan_id"),
}

MEASURE_FACT: tuple[str, ...] = (
    "jumlah_kendaraan_total",
    "jumlah_tidak_daftar_ulang",
    "jumlah_belum_daftar_ulang",
    "jumlah_daftar_ulang",
    "tingkat_kepatuhan",
    "estimasi_pajak_hilang",
)


class ValidasiGagalError(RuntimeError):
    pass


@dataclass
class LaporanValidasi:
    fatal: list[str] = field(default_factory=list)
    peringatan: list[str] = field(default_factory=list)
    info: list[str] = field(default_factory=list)
    metrik: dict[str, Any] = field(default_factory=dict)

    @property
    def lolos(self) -> bool:
        return not self.fatal

    def tambah_fatal(self, pesan: str) -> None:
        self.fatal.append(pesan)

    def tambah_peringatan(self, pesan: str) -> None:
        self.peringatan.append(pesan)

    def tambah_info(self, pesan: str) -> None:
        self.info.append(pesan)

    def ringkasan(self) -> str:
        garis = "=" * 78
        baris = [garis, "RINGKASAN VALIDASI", garis]

        baris.append("")
        baris.append("Jumlah baris per tabel:")
        for nama, n in (self.metrik.get("jumlah_baris") or {}).items():
            baris.append(f"  - {nama:<32} {n:>10,} baris")

        if null_measure := self.metrik.get("null_per_measure"):
            baris.append("")
            baris.append("Null per measure (fakta_kepatuhan_kendaraan):")
            for kolom, n in null_measure.items():
                baris.append(f"  - {kolom:<32} {n:>10,} null")

        if self.info:
            baris.append("")
            baris.append("Catatan proses:")
            baris += [f"  - {p}" for p in self.info]

        baris.append("")
        if self.peringatan:
            baris.append(f"PERINGATAN ({len(self.peringatan)}):")
            baris += [f"  ! {p}" for p in self.peringatan]
        else:
            baris.append("PERINGATAN: tidak ada.")

        baris.append("")
        if self.fatal:
            baris.append(f"FATAL ({len(self.fatal)}):")
            baris += [f"  X {p}" for p in self.fatal]
            baris.append("")
            baris.append("STATUS: GAGAL -- output tidak boleh dipakai untuk dashboard.")
        else:
            baris.append("STATUS: LOLOS -- star schema siap dipakai.")
        baris.append(garis)
        return "\n".join(baris)


def cek_integritas_fk(star: StarSchema, laporan: LaporanValidasi) -> None:
    fact = star.fact
    for fk, (nama_dim, kolom_dim) in RELASI_FK.items():
        if fk not in fact.columns:
            laporan.tambah_fatal(f"Kolom FK '{fk}' tidak ada di fact.")
            continue
        dim = star.tabel.get(nama_dim)
        if dim is None:
            laporan.tambah_fatal(f"Tabel dimensi '{nama_dim}' tidak terbangun.")
            continue

        n_null = int(fact[fk].isna().sum())
        if n_null:
            laporan.tambah_fatal(
                f"{n_null:,} baris fact punya '{fk}' null (gagal join ke {nama_dim})."
            )
        valid = set(dim[kolom_dim].dropna().tolist())
        yatim = sorted({v for v in fact[fk].dropna().tolist() if v not in valid})
        if yatim:
            laporan.tambah_fatal(
                f"{len(yatim):,} nilai '{fk}' tidak ada di {nama_dim} (orphan FK): "
                f"{yatim[:10]}"
            )


def cek_rentang_kepatuhan(star: StarSchema, laporan: LaporanValidasi) -> None:
    fact = star.fact
    if "tingkat_kepatuhan" not in fact.columns:
        laporan.tambah_fatal("Kolom 'tingkat_kepatuhan' tidak ada di fact.")
        return
    nilai = pd.to_numeric(fact["tingkat_kepatuhan"], errors="coerce")
    diluar = fact[nilai.notna() & ((nilai < 0) | (nilai > 1))]
    if not diluar.empty:
        contoh = diluar.head(5)[list(GRAIN_FACT) + ["tingkat_kepatuhan"]]
        laporan.tambah_fatal(
            f"{len(diluar):,} baris punya tingkat_kepatuhan di luar [0,1].\n"
            f"{contoh.to_string(index=False)}"
        )


def cek_duplikat_grain(star: StarSchema, laporan: LaporanValidasi) -> None:
    fact = star.fact
    grain = [k for k in GRAIN_FACT if k in fact.columns]
    if len(grain) != len(GRAIN_FACT):
        laporan.tambah_fatal(
            f"Kolom grain tidak lengkap di fact; ditemukan {grain}, harusnya {list(GRAIN_FACT)}."
        )
        return
    dup = fact[fact.duplicated(subset=grain, keep=False)]
    if not dup.empty:
        laporan.tambah_fatal(
            f"{len(dup):,} baris fact duplikat pada grain {tuple(grain)}.\n"
            f"{dup.head(5)[grain].to_string(index=False)}"
        )


def cek_rentang_tahun(
    star: StarSchema, konfig: Mapping[str, Any], laporan: LaporanValidasi
) -> None:
    meta = (konfig or {}).get("meta", {}) or {}
    fact = star.fact
    tahun = pd.to_numeric(fact["tahun"], errors="coerce").dropna()
    if tahun.empty:
        laporan.tambah_fatal("Fact tidak punya satu pun tahun valid.")
        return

    minimum, maksimum = int(tahun.min()), int(tahun.max())
    laporan.metrik["rentang_tahun"] = (minimum, maksimum)
    laporan.tambah_info(
        f"rentang tahun pada fact: {minimum}-{maksimum} ({tahun.nunique()} tahun unik)"
    )

    harap_min, harap_maks = meta.get("tahun_min"), meta.get("tahun_max")
    if harap_min is not None and minimum != int(harap_min):
        laporan.tambah_peringatan(
            f"tahun minimum {minimum} != ekspektasi {harap_min} (config/sources.yaml meta.tahun_min)."
        )
    if harap_maks is not None and maksimum != int(harap_maks):
        laporan.tambah_peringatan(
            f"tahun maksimum {maksimum} != ekspektasi {harap_maks} (config/sources.yaml meta.tahun_max)."
        )


def cek_kelengkapan_measure(star: StarSchema, laporan: LaporanValidasi) -> None:
    fact = star.fact
    null_measure = {
        kolom: int(fact[kolom].isna().sum())
        for kolom in MEASURE_FACT
        if kolom in fact.columns
    }
    laporan.metrik["null_per_measure"] = null_measure

    for kolom, n in null_measure.items():
        if n:
            persen = n / max(len(fact), 1) * 100
            laporan.tambah_peringatan(
                f"{n:,} null ({persen:.1f}%) pada measure '{kolom}'."
            )

    n_floor = int(star.metrik.get("baris_difloor", 0) or 0)
    if n_floor:
        laporan.tambah_peringatan(
            f"{n_floor:,} baris di-floor ke 0 pada 'jumlah_daftar_ulang' "
            "(total - tidak - belum bernilai negatif)."
        )

    if "estimasi_pajak_hilang" in fact.columns:
        total_hilang = float(
            pd.to_numeric(fact["estimasi_pajak_hilang"], errors="coerce").sum(skipna=True)
        )
        laporan.metrik["total_estimasi_pajak_hilang"] = total_hilang
        laporan.tambah_info(
            f"total estimasi_pajak_hilang seluruh periode: Rp {total_hilang:,.0f}"
        )

    if "tingkat_kepatuhan" in fact.columns:
        rata = pd.to_numeric(fact["tingkat_kepatuhan"], errors="coerce").mean()
        if pd.notna(rata):
            laporan.metrik["rata_tingkat_kepatuhan"] = float(rata)
            laporan.tambah_info(f"rata-rata tingkat_kepatuhan (unweighted): {rata:.4f}")


def cek_dimensi_tidak_kosong(star: StarSchema, laporan: LaporanValidasi) -> None:
    for nama in ("dim_waktu", "dim_wilayah", "dim_jenis_kendaraan", "dim_fungsi_kendaraan"):
        dim = star.tabel.get(nama)
        if dim is None or dim.empty:
            laporan.tambah_fatal(f"Dimensi '{nama}' kosong.")


def validasi(
    star: StarSchema,
    konfig: Mapping[str, Any] | None = None,
    catatan_tambahan: list[str] | None = None,
) -> LaporanValidasi:
    laporan = LaporanValidasi()
    laporan.metrik["jumlah_baris"] = {k: len(v) for k, v in star.tabel.items()}

    for pesan in list(catatan_tambahan or []) + list(star.catatan):
        laporan.tambah_info(pesan)

    if "fakta_kepatuhan_kendaraan" not in star.tabel:
        laporan.tambah_fatal("Tabel 'fakta_kepatuhan_kendaraan' tidak terbangun.")
        return laporan
    if star.fact.empty:
        laporan.tambah_fatal("Tabel fact kosong (0 baris).")
        return laporan

    cek_dimensi_tidak_kosong(star, laporan)
    cek_integritas_fk(star, laporan)
    cek_rentang_kepatuhan(star, laporan)
    cek_duplikat_grain(star, laporan)
    cek_rentang_tahun(star, konfig or {}, laporan)
    cek_kelengkapan_measure(star, laporan)
    return laporan


def validasi_atau_gagal(
    star: StarSchema,
    konfig: Mapping[str, Any] | None = None,
    catatan_tambahan: list[str] | None = None,
) -> LaporanValidasi:
    laporan = validasi(star, konfig, catatan_tambahan)
    if not laporan.lolos:
        raise ValidasiGagalError(laporan.ringkasan())
    return laporan

## 5 · Export & orkestrasi

`jalankan()` merangkai seluruh tahap: extract → transform → build → validate → export. Bila validasi lolos, semua tabel ditulis ke `data/processed/*.csv` (opsional DuckDB lewat `tulis_duckdb=True`); angka dibulatkan saat ekspor supaya file stabil antar-run. `inspeksi()` membantu memeriksa header CSV mentah sebelum menjalankan pipeline penuh.

In [ ]:
PEMBULATAN_EKSPOR: dict[str, int] = {
    "tingkat_kepatuhan": 6,
    "estimasi_pajak_hilang": 2,
    "rata_pajak_per_kendaraan": 2,
    "nilai_pajak_rupiah": 2,
    "kendaraan_terdaftar": 0,
}

URUTAN_EKSPOR: tuple[str, ...] = (
    "dim_waktu",
    "dim_wilayah",
    "dim_jenis_kendaraan",
    "dim_fungsi_kendaraan",
    "fakta_kepatuhan_kendaraan",
    "bantu_rata_pajak_tahunan",
)


def ekspor_csv(star: StarSchema, dir_keluaran: Path) -> list[Path]:
    dir_keluaran.mkdir(parents=True, exist_ok=True)
    ditulis: list[Path] = []
    for nama in _urutan(star):
        df = _bulatkan(star.tabel[nama])
        path = dir_keluaran / f"{nama}.csv"
        df.to_csv(path, index=False, encoding="utf-8", lineterminator="\n")
        ditulis.append(path)
    return ditulis


def ekspor_duckdb(star: StarSchema, path_db: Path) -> Path:
    try:
        import duckdb
    except ImportError as exc:
        raise SystemExit(
            "Paket 'duckdb' belum terpasang. Pasang dulu:\n"
            "    pipenv install --categories duckdb\n"
            "atau jalankan tanpa flag --duckdb."
        ) from exc

    path_db.parent.mkdir(parents=True, exist_ok=True)
    if path_db.exists():
        path_db.unlink()

    koneksi = duckdb.connect(str(path_db))
    try:
        for nama in _urutan(star):
            df = _bulatkan(star.tabel[nama])
            koneksi.register("df_sementara", df)
            koneksi.execute(f'CREATE OR REPLACE TABLE "{nama}" AS SELECT * FROM df_sementara')
            koneksi.unregister("df_sementara")
    finally:
        koneksi.close()
    return path_db


def _urutan(star: StarSchema) -> list[str]:
    sisa = [k for k in star.tabel if k not in URUTAN_EKSPOR]
    return [k for k in URUTAN_EKSPOR if k in star.tabel] + sorted(sisa)


def _bulatkan(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for kolom, digit in PEMBULATAN_EKSPOR.items():
        if kolom in df.columns:
            df[kolom] = pd.to_numeric(df[kolom], errors="coerce").round(digit)
    return df


def inspeksi(konfig: Mapping[str, Any], root: Path) -> int:
    peta_alias = bangun_peta_alias(konfig.get("alias_kolom", {}))
    semua_sumber = {**konfig.get("sumber", {}), **(konfig.get("sumber_opsional") or {})}
    meta = konfig.get("meta", {}) or {}

    print("=" * 78)
    print("INSPEKSI HEADER CSV MENTAH (data/raw/)")
    print("=" * 78)
    ada_file = False

    for alias, ksum in semua_sumber.items():
        path = path_sumber(ksum, root)
        print(f"\n### {alias}  ({ksum.get('file')})")
        if not path.exists():
            wajib = "WAJIB" if ksum.get("wajib", True) else "opsional"
            print(f"    [belum ada] file {wajib} ini belum diletakkan di data/raw/.")
            continue

        ada_file = True
        df = baca_csv(path, str(meta.get("encoding", "utf-8")), meta.get("sep", "auto"))
        print(f"    baris: {len(df):,} | kolom: {len(df.columns)}")
        print(f"    header asli : {list(df.columns)}")

        ternormalisasi = normalisasi_nama_kolom(df)
        terpetakan = petakan_kolom(ternormalisasi, peta_alias, ksum.get("rename"))
        print(f"    setelah map : {list(terpetakan.columns)}")

        baku = set(peta_alias.values()) | {str(ksum.get("measure", ""))}
        sisa = [k for k in terpetakan.columns if k not in baku]
        if sisa:
            print(f"    belum dikenali (kandidat kolom kategori/measure): {sisa}")
        print(f"    contoh baris:\n{df.head(3).to_string(index=False)}")

    print("\n" + "=" * 78)
    if not ada_file:
        print("Tidak ada satu pun CSV di data/raw/. Lihat README bagian 'Menyiapkan data'.")
    else:
        print("Kalau ada kolom penting yang 'belum dikenali', tambahkan namanya ke")
        print("config/sources.yaml -> alias_kolom (atau rename pada sumber terkait).")
    print("=" * 78)
    return 0


def jalankan(
    path_konfig: Path | None = None,
    dir_keluaran: Path | None = None,
    root: Path | None = None,
    tulis_duckdb: bool = False,
    diam: bool = False,
) -> tuple[StarSchema, LaporanValidasi]:
    root = root or ROOT_PROYEK
    dir_keluaran = dir_keluaran or DIR_PROCESSED_DEFAULT
    konfig = muat_konfigurasi(path_konfig or KONFIG_DEFAULT)

    def log(pesan: str = "") -> None:
        if not diam:
            print(pesan)

    log("=" * 78)
    log("ETL STAR SCHEMA KEPATUHAN PAJAK KENDARAAN BERMOTOR - JAWA BARAT")
    log("=" * 78)

    log("\n[1/5] EXTRACT - membaca CSV mentah dari data/raw/ (tanpa akses jaringan)")
    tabel_mentah = ekstrak(konfig, root)
    log(ringkas_ekstrak(tabel_mentah, root))

    log("\n[2/5] TRANSFORM - membersihkan, unpivot, menyamakan grain")
    hasil = transform_semua(tabel_mentah, konfig)
    catatan_transform = kumpulkan_catatan(hasil)
    for pesan in catatan_transform:
        log(f"  {pesan}")

    log("\n[3/5] BUILD - membangun dimensi, surrogate key, dan measure turunan")
    star = bangun_star(hasil, konfig)
    for pesan in star.catatan:
        log(f"  {pesan}")

    log("\n[4/5] VALIDATE")
    laporan = validasi(star, konfig, catatan_transform)

    log("\n[5/5] EXPORT")
    if laporan.lolos:
        for path in ekspor_csv(star, dir_keluaran):
            log(f"  ditulis: {_rel(path, root)}")
        if tulis_duckdb:
            path_db = ekspor_duckdb(star, dir_keluaran / "star.duckdb")
            log(f"  ditulis: {_rel(path_db, root)}")
    else:
        log("  DILEWATI - validasi menemukan masalah FATAL, output tidak ditulis.")

    log("")
    log(laporan.ringkasan())
    _simpan_laporan(laporan, dir_keluaran)
    return star, laporan


def _simpan_laporan(laporan: LaporanValidasi, dir_keluaran: Path) -> None:
    dir_keluaran.mkdir(parents=True, exist_ok=True)
    (dir_keluaran / "laporan_validasi.txt").write_text(
        laporan.ringkasan() + "\n", encoding="utf-8"
    )


def _rel(path: Path, root: Path) -> str:
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return path.as_posix()

## Jalankan pipeline

Pastikan CSV wajib sudah ada di `data/raw/`. Sel di bawah menjalankan seluruh alur dan mencetak ringkasan validasi. Hasilnya tersedia pada variabel `star` (semua tabel) dan `laporan` (ringkasan validasi).

In [ ]:
star, laporan = jalankan(tulis_duckdb=False, diam=False)